# Publication preservation checks

This notebook checks unchanged scientific files and notebook calculation cells after publication path-label normalization. Historical source hashes and current publication hashes remain separate. The portable evidence notebook records the bounded fit; no full benchmark is rerun here.

In [1]:
from pathlib import Path
import hashlib, json, platform, sys, zipfile, nbformat
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'benchmarks/curation/publication_scientific_baseline.json').is_file())
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''):h.update(block)
    return h.hexdigest()
runtime={'environment_name':'chronoclone','absolute_macbook_path':'/path/to/ChronoClone/.runtime/chronoclone','kernel':'chronoclone-paper','current_prefix':sys.prefix,'current_executable':sys.executable,'python':platform.python_version(),'versions':{'nbformat':nbformat.__version__,'pandas':pd.__version__},'execution_command':'JUPYTER_PATH=/path/to/ChronoClone/.runtime/jupyter python benchmarks/run_notebook.py benchmarks/curation/0003_publication_preservation.ipynb --kernel chronoclone-paper'}
display(pd.Series(runtime,name='Execution environment').to_frame())
baseline=json.loads((ROOT/'benchmarks/curation/publication_scientific_baseline.json').read_text())

,Execution environment
environment_name,chronoclone
absolute_macbook_path,/path/to/ChronoClone/.runt...
kernel,chronoclone-paper
current_prefix,/path/to/ChronoClone/.runt...
current_executable,/path/to/ChronoClone/.runt...
python,3.11.15
versions,"{'nbformat': '5.10.4', 'pandas': '2.3.3'}"
execution_command,JUPYTER_PATH=~/Documents/Chro...


In [2]:
records=[]
for row in baseline['scientific_files']:
    p=ROOT/row['path']
    passed=p.is_file() and p.stat().st_size==row['bytes'] and sha(p)==row['sha256']
    assert passed,row['path']
    records.append({'path':row['path'],'bytes':row['bytes'],'passed':passed})
for row in baseline['scientific_cells']:
    nb=nbformat.read(ROOT/row['path'],as_version=4)
    assert hashlib.sha256(nb.cells[row['cell_index']].source.encode()).hexdigest()==row['sha256']
frame=pd.DataFrame(records)
display(frame.assign(format=frame.path.map(lambda p:Path(p).suffix)).groupby('format').agg(files=('path','size'),bytes=('bytes','sum'),passed=('passed','sum')))
print('Unchanged scientific code cells:',len(baseline['scientific_cells']))

,files,bytes,passed
format,,,
.csv,12,18643041,12
.gz,1,153477867,1
.json,3,11182,3
.png,14,1839459,14
.py,2,59084,2
.svg,30,12425615,30
.whl,1,29428,1


Unchanged scientific code cells: 37


In [3]:
notebooks=[]
for name in ['0003_t1_locked_method_noise_results.ipynb','0005_t1_locked_four_method_noise_results.ipynb','0006_benchmark_evidence_and_smoke.ipynb']:
    p=ROOT/'benchmarks/ode_noise/notebooks'/name
    nb=nbformat.read(p,as_version=4);nbformat.validate(nb)
    codes=[c for c in nb.cells if c.cell_type=='code']
    assert [c.execution_count for c in codes]==list(range(1,len(codes)+1))
    assert not [o for c in codes for o in c.outputs if o.output_type=='error']
    notebooks.append({'path':str(p.relative_to(ROOT)),'code_cells':len(codes),'output_cells':sum(bool(c.outputs) for c in codes),'publication_sha256':sha(p)})
wheel=next((ROOT/'benchmarks/frozen_runtime').rglob('*.whl'))
with zipfile.ZipFile(wheel) as z:
    assert z.testzip() is None
    wheel_members=len(z.namelist())
summary={'status':'pass','scientific_files':len(records),'scientific_cells':len(baseline['scientific_cells']),'notebooks':notebooks,'wheel_members':wheel_members,'runtime':runtime,'full_benchmark_rerun':False}
(ROOT/'benchmarks/curation/publication_preservation_validation.json').write_text(json.dumps(summary,indent=2)+'\n')
display(pd.DataFrame(notebooks))
print('Wheel archive members:',wheel_members)

,path,code_cells,output_cells,publication_sha256
0,benchmarks/ode_noise/notebooks/0003_t1_locked_...,20,20,317148369f9f6af44b08082650fd15530ed6d37602da86...
1,benchmarks/ode_noise/notebooks/0005_t1_locked_...,19,19,a7dc4f3ae963d1893c7bfb59998319df9135ce1087e935...
2,benchmarks/ode_noise/notebooks/0006_benchmark_...,5,5,771c534fa36bec61c3a5efa339ee511a774ea781a1c42e...


Wheel archive members: 14
